## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH05/ch05_tora.ipynb)                                             

# About this Notebook

This notebook demonstrates how to generate **trajectory-guided videos** using **Tora**, a trajectory-oriented diffusion framework developed by Alibaba's Research & Intelligence Computing team. Tora extends conventional text-to-video models by introducing **spatial control via 2D trajectory prompts**, enabling users to influence the motion path of generated objects in video clips.

### Steps Included:

1. **Repository Setup and Environment Configuration**:
   The Tora GitHub repository is cloned and the working directory is changed to the `sat/` subdirectory where the codebase resides. Required dependencies are installed, including `huggingface_hub` with fast transfer enabled to streamline model checkpoint downloads.

2. **Model Checkpoint Download**:
   The pretrained weights for **CogVideoX-5B** fine-tuned on Tora (a sparse trajectory-conditioned diffusion model) are downloaded directly from the Hugging Face Hub. These checkpoints are stored in the local `ckpts/tora/t2v` directory.

3. **Inference with Trajectory Guidance**:
   The main generation script `sample_video.py` is executed using `torchrun` with one GPU. It reads:

   * A **text prompt** file that defines the visual concept (e.g., "a drone flying over a forest").
   * A **trajectory file** (`trajs/coaster.txt`) that defines a 2D motion path.
     The script generates videos consistent with both the text and trajectory prompt and stores them in the `samples` directory.

4. **Interactive App Launch**:
   A Gradio-based web interface is started using `app.py`, allowing users to upload their own prompts and trajectories to interactively generate videos without modifying the source code.

### Highlights:

* Combines **language-based semantics** with **trajectory control**, making video generation more precise and expressive.
* Enables experimentation with **sparse trajectory inputs**, offering better generalization and fewer constraints than dense trajectory supervision.
* Supports **multi-modal input**, integrating language and spatial motion cues into the video generation process.


# Installs

In [1]:
# Clone this repository.
!git clone https://github.com/Nicolepcx/Tora
%cd Tora

/content/Tora


In [2]:
!ls

CogVideoX_LICENSE  LICENSE  pyproject.toml  sat
diffusers-version  modules  README.md


In [3]:
%cd sat/

/content/Tora/sat


In [4]:
!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.3/812.3 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.3/833.3 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/917.8 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 141.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.6 MB/s eta 0:00:00
 

In [5]:
# `torchvision.io.write_video` was deprecated in torchvision 0.22 and removed in 0.24, so torchvision needs to be downgraded.
!pip install -q "torchvision<0.24"
!pip install "huggingface_hub==1.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 16.6 MB/s eta 0:00:00


In [6]:
!pip uninstall -y torchaudio
!pip install torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 70.5 MB/s eta 0:00:00


In [7]:
!pip uninstall -y gradio gradio-client fastapi starlette pandas numpy

!pip install --force-reinstall --no-cache-dir \
    "numpy==1.26.4" \
    "pandas==2.2.3" \
    "pydantic==2.5.3" \
    "gradio==3.50.2" \
    "gradio-client==0.6.1" \
    "fastapi==0.104.1" \
    "starlette==0.27.0"

Found existing installation: gradio 3.50.2
Uninstalling gradio-3.50.2:
  Successfully uninstalled gradio-3.50.2
Found existing installation: gradio_client 0.6.1
Uninstalling gradio_client-0.6.1:
  Successfully uninstalled gradio_client-0.6.1
Found existing installation: fastapi 0.104.1
Uninstalling fastapi-0.104.1:
  Successfully uninstalled fastapi-0.104.1
Found existing installation: starlette 0.27.0
Uninstalling starlette-0.27.0:
  Successfully uninstalled starlette-0.27.0
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 86.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 315.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 286.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 

In [7]:
#!HF_XET_HIGH_PERFORMANCE=1 hf download Alibaba-Research-Intelligence-Computing/Tora --local-dir ckpts
!hf download Alibaba-Research-Intelligence-Computing/Tora --local-dir ckpts

Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.23.0.
To update, run: hf update
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0% 0/16 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.99G [00:00<?, ?B/s]         Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/4.99G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 36.8k/9.52G [00:00<10:41:43, 247kB/s]
Reconstructing (incomplete total...):   0% 36.8k/9.52G [00:00<10:41:43, 247kB/s]
Reconstructing (incomplete total...):   0% 39.4k/9.52G [00:00<10:41:43, 247kB/s]
Reconstructing (incomplete total...):   0% 49.0k/9.52G [00:00<10:41:43, 247kB/s]

Fetching 16 files:   6% 1/16 [00:00<00:03,  4.72it/s]
Reconstructing (incomplete total...):   0% 50.5k/9.53G [00:00<10:41:46, 247kB/s]
Reconstru

# Run inference

In [8]:
!N_GPU=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True torchrun --standalone --nproc_per_node=1 sample_video.py --base configs/tora/model/cogvideox_5b_tora.yaml configs/tora/inference_sparse.yaml --load ckpts/tora/t2v --output-dir samples --point_path trajs/coaster.txt --input-file assets/text/t2v/examples.txt

[2026-08-18 06:06:51,729] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
/content/Tora/sat/sgm/modules/encoders/modules.py:246: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if model_dir is not "google/t5-v1_1-xxl":
/usr/local/lib/python3.12/dist-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
no module 'xformers'. Processing without...
no module 'xformers'. Processing without...
[2026-08-18 06:07:07,146] [WARNING] No training data specified
[2026-08-18 06:07:07,146] [WARNING] No train_iters (recommended) or epochs specified, use default 10k iters.
[2026-08-18 06:07:07,148] [INFO] using world size: 1
[W818 06:07:07.357867244 Utils.hpp:166] Warning: Environment variable NCCL_BLOCKING_WAIT is deprecated; use TORCH_NCCL_BLOCKING_WAIT inste

# Start Gradio app

In [ ]:
!python app.py --load ckpts/tora/t2v

/usr/local/lib/python3.12/dist-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
no module 'xformers'. Processing without...
no module 'xformers'. Processing without...
[2026-08-18 06:40:11,039] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-08-18 06:40:13,697] [WARNING] No training data specified
[2026-08-18 06:40:13,697] [WARNING] No train_iters (recommended) or epochs specified, use default 10k iters.
[2026-08-18 06:40:13,697] [INFO] using world size: 1
[W818 06:40:13.890629223 Utils.hpp:166] Warning: Environment variable NCCL_BLOCKING_WAIT is deprecated; use TORCH_NCCL_BLOCKING_WAIT instead (function operator())
[2026-08-18 06:40:13,944] [INFO] [RANK 0] > initializing model parallel with size 1
[2026-08-18 06:40:13,945] [INFO] [RANK 0] You didn't 